# 07. メンテナンス（Spark）

Iceberg は書き込みのたびに新しいファイルとスナップショットを作り、古いものを消しません。
おかげでタイムトラベルができますが、放っておくと次のものが溜まっていきます。

| 溜まるもの | 困ること | 対処（Spark のプロシージャ） |
| --- | --- | --- |
| 小さなデータファイル | クエリが遅くなる | `rewrite_data_files`（Compaction） |
| マニフェスト（ファイル一覧のメタデータ） | クエリの計画が遅くなる | `rewrite_manifests` |
| 古いスナップショット | 参照しているファイルを消せず、ストレージを圧迫する | `expire_snapshots` |
| どこからも参照されないファイル | ストレージの無駄 | `remove_orphan_files` |

- テーブル: `handson.maint_spark`

## 準備: SparkSession を作る

接続設定は `spark-defaults.conf` にあるので、ここでは何も指定しません（01 と同じ）。

In [ ]:
from decimal import Decimal
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("07_maintenance").getOrCreate()

def sql(query):
    """SQL を実行し、結果があれば表示する"""
    df = spark.sql(query)
    if df.columns:
        df.show(truncate=False)

## 1. 小さな書き込みを 10 回繰り返す

1 回の INSERT ごとにデータファイルとスナップショットが 1 つずつ増えます。
状態を見るための関数も用意します。

In [ ]:
sql("DROP TABLE IF EXISTS handson.maint_spark PURGE")
sql("CREATE TABLE handson.maint_spark (trip_id BIGINT, vendor STRING, fare DECIMAL(10, 2)) USING iceberg")

for i in range(1, 11):
    sql(f"INSERT INTO handson.maint_spark VALUES ({i}, 'V{i % 3}', {i * 1.5})")

def status():
    """現在のファイル数、全スナップショットが参照するファイル数、マニフェスト数、スナップショット数"""
    sql("""
    SELECT
      (SELECT count(*) FROM handson.maint_spark.files)           AS current_data_files,
      (SELECT count(DISTINCT file_path) FROM handson.maint_spark.all_data_files) AS all_data_files,
      (SELECT count(*) FROM handson.maint_spark.manifests)       AS current_manifests,
      (SELECT count(*) FROM handson.maint_spark.snapshots)       AS snapshots
    """)

status()

- `current_data_files`: 現在のスナップショットが参照するデータファイル
- `all_data_files`: 残っているすべてのスナップショットが参照するデータファイル（= ストレージに残っている分）

## 2. Compaction（rewrite_data_files）

小さなファイルを読み、まとめて書き直します。現在のファイルは 1 つになりますが、
古いファイルは過去のスナップショットから参照されているので、まだストレージに残っています（`all_data_files` は増える）。

In [ ]:
sql("CALL lakehouse.system.rewrite_data_files(table => 'lakehouse.handson.maint_spark')")
status()

## 3. マニフェストの統合（rewrite_manifests）

マニフェストは「どのデータファイルがテーブルに属するか」を記録したメタデータのファイルです。
書き込みを繰り返すと増えていくので、まとめ直します。

In [ ]:
sql("CALL lakehouse.system.rewrite_manifests('lakehouse.handson.maint_spark')")
status()

## 4. スナップショットの失効（expire_snapshots）

古いスナップショットを削除し、**どのスナップショットからも参照されなくなったファイルを実際に削除**します。
既定では 5 日より古いものだけが対象なので、ここでは「今より古いものすべて。ただし最新の 1 つは残す」と指定します。

In [ ]:
from datetime import datetime

oldest_id = spark.sql("SELECT snapshot_id FROM handson.maint_spark.snapshots ORDER BY committed_at LIMIT 1").first()[0]
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

sql(f"CALL lakehouse.system.expire_snapshots(table => 'lakehouse.handson.maint_spark', older_than => TIMESTAMP '{now}', retain_last => 1)")
status()

`all_data_files` が `current_data_files` と同じになり、古いファイルがストレージから消えました。
その代わり、失効したスナップショットにはもうタイムトラベルできません。**タイムトラベルできる期間とストレージの量はトレードオフ** です。

In [ ]:
try:
    sql(f"SELECT * FROM handson.maint_spark VERSION AS OF {oldest_id}")
except Exception as e:
    print(type(e).__name__, str(e).splitlines()[0])

## 5. 孤立ファイルの削除（remove_orphan_files）

書き込みが途中で失敗すると、どのスナップショットからも参照されないファイル（孤立ファイル）が残ることがあります。
ここでは PyIceberg を使い、テーブルの `data/` にわざと孤立ファイルを置きます。

In [ ]:
from pyiceberg.catalog import load_catalog

table = load_catalog("lakehouse").load_table("handson.maint_spark")
orphan = f"{table.location()}/data/orphan-by-handson.parquet"
with table.io.new_output(orphan).create(overwrite=True) as f:
    f.write(b"not referenced by any snapshot")
print("置いたファイル:", orphan)

`remove_orphan_files` はテーブルの場所にあるファイルを一覧し、メタデータから参照されていないものを削除します。
書き込み中のファイルを誤って消さないよう、SQL のプロシージャは **24 時間より新しいファイルを対象にできません**（既定は 3 日）。

まずプロシージャで、今置いたばかりのファイルが対象にならないことを確かめます。

`prefix_listing => true` は、ファイルの一覧に Hadoop のファイルシステムではなく Iceberg の FileIO（Polaris から払い出された認証情報を使う）を使う指定です。この環境の Spark には S3 用の Hadoop ファイルシステムを入れていないので必要です。

In [ ]:
sql("CALL lakehouse.system.remove_orphan_files(table => 'lakehouse.handson.maint_spark', dry_run => true, prefix_listing => true)")

何も表示されません（3 日より古い孤立ファイルはないため）。

ハンズオンでは今置いたファイルを消したいので、プロシージャの代わりに Java の **Action API** を PySpark から呼び、期間の制限なしで実行します。
本番では他の書き込みと衝突する危険があるので、短い期間での実行は避けます。

In [ ]:
import time

jvm = spark._jvm
jtable = jvm.org.apache.iceberg.spark.Spark3Util.loadIcebergTable(spark._jsparkSession, "lakehouse.handson.maint_spark")
result = (
    jvm.org.apache.iceberg.spark.actions.SparkActions.get(spark._jsparkSession)
    .deleteOrphanFiles(jtable)
    .olderThan(int(time.time() * 1000))  # 今より古いファイルすべて
    .usePrefixListing(True)              # Iceberg の FileIO でファイルを一覧する
    .execute()
)
for location in result.orphanFileLocations():
    print("削除した孤立ファイル:", location)

## まとめ

- Compaction（`rewrite_data_files`）で小さなファイルをまとめる。古いファイルは過去のスナップショットのために残る
- `rewrite_manifests` でメタデータのファイルを整理する
- `expire_snapshots` で古いスナップショットを消すと、はじめてストレージからファイルが消える。消したスナップショットにはタイムトラベルできなくなる
- `remove_orphan_files` で、どこからも参照されないファイルを消す
- 実運用では、これらを定期的に実行する（02 で付けたタグのスナップショットは `expire_snapshots` の対象にならない）